# Instructor Setup — Robust AI Data Platforms Workshop

Run from **`/Workspace/Shared/Robust-AI-Data-Platforms-workshop/instructor/scripts/00_instructor_setup`**

**Run this notebook once before the workshop begins. Requires workspace ADMIN.**

This notebook:
1. Creates the shared schema and Volumes in the `platform-workshop` catalog
2. Grants participant permissions to account users
3. Guides you through uploading the financial document corpus to the shared Volume
4. Deploys the Databricks Asset Bundle (pipeline + orchestration job)
5. Runs the pipeline to populate silver, gold, and search tables
6. Creates the Vector Search endpoint and Delta Sync index (Session 2)
7. Creates the Agent Bricks Knowledge Assistant (Session 2)
8. Deploys the Graph Explorer app (Session 2)

**Prerequisites:**
- WORKSPACE ADMIN role
- Databricks CLI installed and authenticated (`databricks auth login`)
- Financial PDFs downloaded from Google Drive (see Step 3)

In [0]:
# The knowledge_assistants module requires databricks-sdk >= 0.100.0
%pip install "databricks-sdk>=0.100.0" --quiet
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "platform-workshop", "Catalog")

catalog       = dbutils.widgets.get("catalog")
shared_schema = "00_shared"
c = f"`{catalog}`"

print(f"Catalog:       {catalog}")
print(f"Shared schema: {shared_schema}")

## Step 1: Create the shared schema and Volumes

In [0]:
%sql
USE CATALOG ${catalog}

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS 00_shared
COMMENT 'Shared schema for the Robust AI Data Platforms workshop. Contains pre-staged financial PDFs and pipeline outputs.';

In [0]:
%sql
-- Main corpus: all financial PDFs (10-K, 10-Q, earnings calls, annual reports)
CREATE VOLUME IF NOT EXISTS 00_shared.financial_docs_raw
  COMMENT 'Full financial document corpus: Apple, Amazon, Microsoft, NVIDIA, Meta';

-- Small sample for participant upload exercise (2–3 PDFs)
CREATE VOLUME IF NOT EXISTS 00_shared.financial_docs_sample
  COMMENT '2–3 sample PDFs for the participant manual upload exercise';

## Step 2: Grant permissions to workshop participants

Grants access to all authenticated users in the account (`account users`).
Participants also need CREATE SCHEMA on the catalog to create their personal schema.

In [0]:
%sql
GRANT USE CATALOG ON CATALOG ${catalog} TO `account users`;
GRANT CREATE SCHEMA ON CATALOG ${catalog} TO `account users`;

GRANT USE SCHEMA, SELECT ON SCHEMA ${catalog}.00_shared TO `account users`;

GRANT READ VOLUME ON VOLUME ${catalog}.00_shared.financial_docs_raw    TO `account users`;
GRANT READ VOLUME ON VOLUME ${catalog}.00_shared.financial_docs_sample TO `account users`;

SELECT 'Permissions granted' AS status;

## Step 3: Upload financial documents to the shared Volume


**Original Google Drive source:**
https://drive.google.com/drive/u/0/folders/1cYaK_PeQ7O_BbslTMMkR7VAuxwci3kMD


In [0]:
import shutil

def copy_docs(subfolder):
    # Derive the bundle root from this notebook's own workspace path.
    # Works regardless of which user deployed the bundle or where it was synced.
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    # e.g. /Users/dennis.schultz/vectorMLWorkshop/src/utils/01_admin_setup
    bundle_root = "/Workspace" + notebook_path.rsplit("/scripts/", 1)[0]
    artifact_src = f"{bundle_root}/artifacts"

    shutil.copytree(
    f"{artifact_src}/financial_docs{subfolder}",
    f"/Volumes/{catalog}/00_shared/financial_docs_raw{subfolder}",
    dirs_exist_ok=True
    )

    print(f"Copied documents to /Volumes/{catalog}/00_shared/financial_docs_raw{subfolder}")

In [0]:
import shutil
import os

# Derive the bundle root from this notebook's own workspace path.
# Works regardless of which user deployed the bundle or where it was synced.
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# e.g. /Users/dennis.schultz/vectorMLWorkshop/src/utils/01_admin_setup
bundle_root = "/Workspace" + notebook_path.rsplit("/scripts/", 1)[0]
artifact_src = f"{bundle_root}/artifacts/samples"
dest_dir = f"/Volumes/{catalog}/00_shared/financial_docs_sample"

os.makedirs(dest_dir, exist_ok=True)
for file_name in os.listdir(artifact_src):
    src_file = os.path.join(artifact_src, file_name)
    dest_file = os.path.join(dest_dir, file_name)
    if os.path.isfile(src_file):
        shutil.copy2(src_file, dest_file)

In [0]:
# Verify the upload — confirm files are in place before proceeding
files = dbutils.fs.ls(f"/Volumes/{catalog}/00_shared/financial_docs_raw/")
total = 0
for f in files:
    if f.isDir():
        company_files = dbutils.fs.ls(f.path)
        count = len([cf for cf in company_files if cf.name.endswith('.pdf')])
        total += count
        print(f"  {f.name.rstrip('/'):<15} {count:>3} PDFs")

print(f"\n  Total: {total} PDFs")

if total < 50:
    print("\n⚠️  Fewer than 50 PDFs found — confirm all companies' documents are uploaded.")
else:
    print("\n✓ Document corpus looks good.")

   
## Step 4: Deploy the Databricks Asset Bundle

1. Navigate to the `participant/bundles/financial_pipelines/` directory and open databricks.yml.
2. Deploy the bundle to **PROD**
3. Run the job using the cells below.  

_Note 1: Do this multiple times while incrementally adding documents to the volume as per Step 3, above._

_Note 2: Verify the catalog variable value in the target in databricks.yml_


This deploys:
- `financial_intelligence_pipeline` (Spark Declarative Pipeline)
- `financial_intelligence_job` (LakeFlow Job)

After deploying, the next cell will locate the job by name so it can be triggered in Step 5:

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Find the job by name
job_name = "financial_intelligence_job"
jobs = [j for j in w.jobs.list(name=job_name)]
if jobs:
    job_id = jobs[0].job_id
    print(f"✓ Job found: {job_id}")
    print(f"  Name: {jobs[0].settings.name}")
else:
    print(f"⚠️  Job '{job_name}' not found — run 'databricks bundle deploy --target prod' first.")

   
## Step 5: Run the job to populate silver, gold, and search tables

This triggers the job and waits for completion.
**Estimated runtime: 30–60 minutes depending on the number of PDFs.**

Run this the evening before the workshop.

In [0]:
 import time
 
 def run_job(job_id):

    # Trigger the job
    run = w.jobs.run_now(job_id=job_id)
    run_id = run.run_id
    print(f"Job run started: {run_id}")

    # Poll for completion
    while True:
        run_status = w.jobs.get_run(run_id=run_id)
        state = run_status.state.life_cycle_state.value
        result = run_status.state.result_state.value if run_status.state.result_state else None
        print(f"  State: {state}" + (f"  Result: {result}" if result else ""))
        if state in ("TERMINATED", "SKIPPED", "INTERNAL_ERROR"):
            break
        time.sleep(30)

    if result == "SUCCESS":
        print("\n✓ Job completed successfully.")
    else:
        print(f"\n✗ Job ended with state: {state}, result: {result}")
        print("  Check the job run output for details.")

In [0]:
copy_docs("/Annual Report")
run_job(job_id)
copy_docs("/10K")
run_job(job_id)
copy_docs("/10Q")
run_job(job_id)
copy_docs("/Call Transcripts")
run_job(job_id)
copy_docs("/Earnings Report")
run_job(job_id)

In [0]:
# Verify output tables
for table in [
    "financial_docs_bronze",
    "financial_docs_silver",
    "financial_docs_gold",
    "company_ai_investment_summary",
    "financial_docs_for_search",   # Search layer — source for Vector Search index
]:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM `{catalog}`.{shared_schema}.{table}").collect()[0]['cnt']
    print(f"  {table:<45} {count:>6,} rows")

## Step 5a: Add comments to all the tables

Must be run after the job completes once.

In [0]:
%sql
    
-- Table comment
ALTER STREAMING TABLE 00_shared.financial_docs_bronze
  SET TBLPROPERTIES ('comment' = 'Raw binary content of financial PDFs ingested from the shared Volume. One row per file. No transformation applied.');

-- Column comments
ALTER STREAMING TABLE 00_shared.financial_docs_bronze
  ALTER COLUMN source_path COMMENT 'Location or path of the original financial PDF file within the shared volume from which the data was ingested.';

ALTER STREAMING TABLE 00_shared.financial_docs_bronze
  ALTER COLUMN content COMMENT 'Raw binary data representing the entire content of the financial PDF file without any processing or transformation.';

ALTER STREAMING TABLE 00_shared.financial_docs_bronze
  ALTER COLUMN file_size_bytes COMMENT 'Size of the financial PDF file measured in bytes, indicating the storage space occupied by the file.';

ALTER STREAMING TABLE 00_shared.financial_docs_bronze
  ALTER COLUMN ingested_at COMMENT 'Timestamp marking when the financial PDF file was ingested into the system, useful for tracking data freshness and processing timelines.';

In [0]:
%sql
    
-- Table comment
ALTER STREAMING TABLE 00_shared.financial_docs_silver
  SET TBLPROPERTIES ('comment' = 'Plain text and AI-extracted structured fields from financial PDFs. Produced by ai_parse_document() and ai_extract().');

-- Column comments
ALTER STREAMING TABLE 00_shared.financial_docs_silver
  ALTER COLUMN source_path COMMENT 'Location or path where the original PDF document is stored, allowing for easy retrieval and reference.';

ALTER STREAMING TABLE 00_shared.financial_docs_silver
  ALTER COLUMN file_size_bytes COMMENT 'Size of the PDF document in bytes, useful for monitoring storage usage and processing times.';

ALTER STREAMING TABLE 00_shared.financial_docs_silver
  ALTER COLUMN ingested_at COMMENT 'Timestamp indicating when the document was imported into the system for parsing and extraction.';

ALTER STREAMING TABLE 00_shared.financial_docs_silver
  ALTER COLUMN plain_text COMMENT 'Full plain text extracted from the financial PDF, serving as the basis for subsequent structured data extraction and embedding.';

ALTER STREAMING TABLE 00_shared.financial_docs_silver
  ALTER COLUMN company COMMENT 'Name of the company associated with the financial document, enabling filtering and grouping by entity.';

ALTER STREAMING TABLE 00_shared.financial_docs_silver
  ALTER COLUMN fiscal_period COMMENT 'Financial period covered by the document, such as a quarter or fiscal year, to contextualize the data.';

ALTER STREAMING TABLE 00_shared.financial_docs_silver
  ALTER COLUMN document_type COMMENT 'Type of financial report or document, for example, balance sheet, income statement, or cash flow statement.';

ALTER STREAMING TABLE 00_shared.financial_docs_silver
  ALTER COLUMN revenue_reported COMMENT 'Revenue figure extracted from the document, representing the total earnings reported for the period.';

ALTER STREAMING TABLE 00_shared.financial_docs_silver
  ALTER COLUMN net_income_reported COMMENT 'Net income value extracted, indicating the profit or loss after expenses for the reported fiscal period.';

In [0]:
%sql
    
-- Table comment
ALTER MATERIALIZED VIEW 00_shared.financial_docs_gold
  SET TBLPROPERTIES ('comment' = 'Classified financial documents with AI investment category and management sentiment. One row per document.');

-- Column comments
ALTER MATERIALIZED VIEW 00_shared.financial_docs_gold
  ALTER COLUMN source_path COMMENT 'Location or path where the original financial document is stored, useful for tracing back to the source file.';

ALTER MATERIALIZED VIEW 00_shared.financial_docs_gold
  ALTER COLUMN file_size_bytes COMMENT 'Size of the document file in bytes, indicating how large the file is.';

ALTER MATERIALIZED VIEW 00_shared.financial_docs_gold
  ALTER COLUMN ingested_at COMMENT 'Timestamp marking when the document was processed and entered into the system.';

ALTER MATERIALIZED VIEW 00_shared.financial_docs_gold
  ALTER COLUMN company COMMENT 'Name of the company to which the financial document pertains.';

ALTER MATERIALIZED VIEW 00_shared.financial_docs_gold
  ALTER COLUMN fiscal_period COMMENT 'The specific financial reporting period the document covers, such as a quarter or fiscal year.';

ALTER MATERIALIZED VIEW 00_shared.financial_docs_gold
  ALTER COLUMN document_type COMMENT 'Classification of the document, such as earnings report, annual report, or other financial disclosures.';

ALTER MATERIALIZED VIEW 00_shared.financial_docs_gold
  ALTER COLUMN revenue_reported COMMENT 'Reported revenue figures as stated in the financial document.';

ALTER MATERIALIZED VIEW 00_shared.financial_docs_gold
  ALTER COLUMN net_income_reported COMMENT 'Reported net income values extracted from the document.';

ALTER MATERIALIZED VIEW 00_shared.financial_docs_gold
  ALTER COLUMN ai_investment_category COMMENT 'Categorization of the company''s AI-related investments based on document content.';

ALTER MATERIALIZED VIEW 00_shared.financial_docs_gold
  ALTER COLUMN management_sentiment COMMENT 'Summary sentiment of the company''s management towards AI investments, derived from the document''s narrative.';

In [0]:
%sql
    
-- Table comment
ALTER MATERIALIZED VIEW 00_shared.company_ai_investment_summary
  SET TBLPROPERTIES ('comment' = 'Aggregated AI investment signals by company, fiscal period, and category. Primary analytical output for workshop exercises.');

-- Column comments
ALTER MATERIALIZED VIEW 00_shared.company_ai_investment_summary
  ALTER COLUMN company COMMENT 'Name of the company to which the AI investment data relates.';

ALTER MATERIALIZED VIEW 00_shared.company_ai_investment_summary
  ALTER COLUMN fiscal_period COMMENT 'Time frame representing the fiscal period for the reported AI investment data.';

ALTER MATERIALIZED VIEW 00_shared.company_ai_investment_summary
  ALTER COLUMN document_type COMMENT 'Type of document from which the AI investment information was extracted, such as reports or filings.';

ALTER MATERIALIZED VIEW 00_shared.company_ai_investment_summary
  ALTER COLUMN ai_investment_category COMMENT 'Classification of the type of AI investment, helping to categorize the nature of the expenditures or initiatives.';

ALTER MATERIALIZED VIEW 00_shared.company_ai_investment_summary
  ALTER COLUMN management_sentiment COMMENT 'Overall tone or attitude of the company''s management towards AI investments as expressed in the analyzed documents.';

ALTER MATERIALIZED VIEW 00_shared.company_ai_investment_summary
  ALTER COLUMN document_count COMMENT 'Number of documents analyzed to generate the investment summary for the given company and period.';

In [0]:
%sql
-- Table comment
ALTER TABLE 00_shared.financial_docs_for_search
  SET TBLPROPERTIES ('comment' = 'Financial documents prepared for Vector Search indexing. Contains plain text and metadata columns for embedding and filtering.');

-- Column comments
ALTER TABLE 00_shared.financial_docs_for_search
  ALTER COLUMN source_path COMMENT 'Path or location where the original financial document file is stored, useful for retrieving the source material.';

ALTER TABLE 00_shared.financial_docs_for_search
  ALTER COLUMN plain_text COMMENT 'Full plain text content extracted from the financial document — embedded by the Vector Search Delta Sync index.';

ALTER TABLE 00_shared.financial_docs_for_search
  ALTER COLUMN company COMMENT 'Name of the company to which the financial document pertains, used as a metadata filter in Vector Search queries.';

ALTER TABLE 00_shared.financial_docs_for_search
  ALTER COLUMN fiscal_period COMMENT 'The specific fiscal period covered by the financial document, used as a metadata filter in Vector Search queries.';

ALTER TABLE 00_shared.financial_docs_for_search
  ALTER COLUMN document_type COMMENT 'Classification of the financial document, used as a metadata filter in Vector Search queries.';

## Step 6: Create the Vector Search endpoint and Delta Sync index (Session 2)

**This step is required only if you are running Session 2.**

The orchestration job produces `financial_docs_for_search` — a plain managed Delta table
with Change Data Feed enabled. This is the source for a Vector Search Delta Sync index,
which the Agent Bricks Knowledge Assistant will use for semantic retrieval.

Why not index `financial_docs_silver` directly?
- `financial_docs_silver` is a Streaming Table; Vector Search Delta Sync requires a regular
  Delta table with CDF enabled — Streaming Tables and Materialized Views are not supported
  as Delta Sync index sources.
- `financial_docs_for_search` is written by the `build_search_table` job task as a plain
  managed Delta table with `delta.enableChangeDataFeed = true`, making it a valid source.

In [0]:
# Make sure Job has been run before creating endpoint and index

import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    EndpointType,
    VectorIndexType,
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    PipelineType,
)

w = WorkspaceClient()

VS_ENDPOINT_NAME = "financial-docs-vs-endpoint"
VS_INDEX_NAME    = f"{catalog}.{shared_schema}.financial_docs_for_search_index"
SOURCE_TABLE     = f"{catalog}.{shared_schema}.financial_docs_for_search"
EMBEDDING_MODEL  = "databricks-gte-large-en"

# ── 6a: Create the Vector Search endpoint (skip if already exists) ─────────
try:
    ep = w.vector_search_endpoints.get_endpoint(endpoint_name=VS_ENDPOINT_NAME)
    print(f"✓ VS endpoint already exists: {VS_ENDPOINT_NAME}  (status: {ep.endpoint_status.state})")
except Exception:
    print(f"Creating VS endpoint: {VS_ENDPOINT_NAME} ...")
    w.vector_search_endpoints.create_endpoint_and_wait(
        name=VS_ENDPOINT_NAME,
        endpoint_type=EndpointType.STANDARD,
    )
    print(f"✓ VS endpoint created: {VS_ENDPOINT_NAME}")

This next cell is likely to time out before the index is finishes syncing.  Initial sync could take an hour.  

Just watch the UI to know when to continue.

In [0]:
# ── 6b: Create the Delta Sync index (skip if already exists) ──────────────
try:
    idx = w.vector_search_indexes.get_index(index_name=VS_INDEX_NAME)
    print(f"✓ VS index already exists: {VS_INDEX_NAME}")
    print(f"  Status: {idx.status.ready}  Indexed rows: {idx.status.indexed_row_count:,}")
except Exception:
    print(f"Creating Delta Sync index: {VS_INDEX_NAME} ...")
    w.vector_search_indexes.create_index(
        name=VS_INDEX_NAME,
        endpoint_name=VS_ENDPOINT_NAME,
        primary_key="source_path",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=SOURCE_TABLE,
            pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name="plain_text",
                    embedding_model_endpoint_name=EMBEDDING_MODEL,
                )
            ],
        ),
    )
    print(f"✓ Delta Sync index created: {VS_INDEX_NAME}")

# ── 6c: Trigger the first sync and wait ───────────────────────────────────
from databricks.sdk.errors import BadRequest

print("\nWaiting for index to be ready for sync ...")
max_retries = 20
for attempt in range(max_retries):
    try:
        w.vector_search_indexes.sync_index(index_name=VS_INDEX_NAME)
        print("Index sync triggered.")
        break
    except BadRequest as e:
        if "not ready" in str(e) and attempt < max_retries - 1:
            print(f"  Index not ready yet, retrying in 30s (attempt {attempt + 1}/{max_retries}) ...")
            time.sleep(30)
        else:
            raise

# Poll until the sync completes (typically 10–30 minutes for ~130 documents)
while True:
    idx = w.vector_search_indexes.get_index(index_name=VS_INDEX_NAME)
    ready = idx.status.ready
    rows  = idx.status.indexed_row_count or 0
    print(f"  ready={ready}  indexed_rows={rows:,}")
    if ready:
        break
    time.sleep(30)

print(f"\n✓ Index sync complete. {rows:,} rows indexed.")
print(f"  Index name for participant notebooks: {VS_INDEX_NAME}")

   
## Step 7: Create the Agent Bricks Knowledge Assistant (Session 2)

**This step is required only if you are running Session 2.**

The Knowledge Assistant is configured against the Vector Search index created in Step 6.
Participants will query it in `session2/02_vector_semantic_indexing.ipynb`.

The next cell uses the Databricks SDK (`w.knowledge_assistants`) to:
1. Create the `financial-intelligence-assistant` Knowledge Assistant
2. Attach the Vector Search index as a knowledge source with `plain_text` as the text column
3. Print the assistant name and endpoint for use in participant notebooks

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.knowledgeassistants import (
    KnowledgeAssistant,
    KnowledgeSource,
    IndexSpec,
)

w = WorkspaceClient()
catalog       = dbutils.widgets.get("catalog")
shared_schema = "00_shared"
VS_INDEX_NAME = f"{catalog}.{shared_schema}.financial_docs_for_search_index"

KA_DISPLAY_NAME = "financial-intelligence-assistant"
KA_DESCRIPTION  = "Answers questions about financial documents (10-K, 10-Q, earnings calls, annual reports) for Apple, Amazon, Microsoft, NVIDIA, and Meta."
KA_INSTRUCTIONS = (
    "Answer questions using only the provided financial documents. "
    "Cite the source document for every claim. "
    "If the documents do not contain enough information, say so."
)

# ── 7a: Create the Knowledge Assistant (skip if already exists) ────────────
existing = [ka for ka in w.knowledge_assistants.list_knowledge_assistants()
            if ka.display_name == KA_DISPLAY_NAME]

if existing:
    created = existing[0]
    print(f"✓ Knowledge Assistant already exists: {created.display_name}")
    print(f"  Name: {created.name}")
else:
    created = w.knowledge_assistants.create_knowledge_assistant(
        knowledge_assistant=KnowledgeAssistant(
            display_name=KA_DISPLAY_NAME,
            description=KA_DESCRIPTION,
            instructions=KA_INSTRUCTIONS,
        )
    )
    print(f"✓ Knowledge Assistant created: {created.display_name}")
    print(f"  Name: {created.name}")

# ── 7b: Attach the Vector Search index as a knowledge source ──────────────
index_source = KnowledgeSource(
    display_name="Financial Documents Index",
    description="Parsed financial documents with embeddings for semantic retrieval",
    source_type="index",
    index=IndexSpec(
        index_name=VS_INDEX_NAME,
        text_col="plain_text",
        doc_uri_col="source_path",
    ),
)

try:
    created_source = w.knowledge_assistants.create_knowledge_source(
        parent=created.name,
        knowledge_source=index_source,
    )
    print(f"✓ Knowledge source attached: {created_source.display_name}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"✓ Knowledge source already attached")
    else:
        raise

print(f"\n  Assistant name for participant notebooks: {created.name}")

## Step 8: Deploy the Graph Explorer app (Session 2)

**This step is required only if you are running Session 2, Lab 4.**

The Graph Explorer is an open-source Databricks App from:
https://github.com/will-yuponce-db/graph-demo

**Deployment steps:**
1. Create a Git folder
2. Replace the value of `warehouse_id` in `config`
3. Deploy as a Databricks App:
   ```bash
   databricks apps deploy graph-explorer
   ```
4. Add `sql` and `model serving` scopes.
5. Note the app SP id.  Give `CAN USE` on SQL Warehouse.
5. Note the app URL and update the `GRAPH_EXPLORER_URL` variable in
 `participant/session2/04_hybrid_context_layers.ipynb`

### Manually Grant permissions
1. Vector Search
   1. Grant CAN USE on the Vector Search Endpoint
1. Agent
   - Apparently, there is no programmatic way to grant permissions on the Agent.  Furthermore, the grant cannot be to the system `all workspace users` group.
   1. Create a group for workshop users
   2. Add everyone to that group
   3. Grant CAN MANAGE to that group on the agent
2. Databricks App
   1. Grant CAN USE on the app to All Workspace Users

## Pre-workshop checklist

Run through this checklist the morning of the workshop:

**Session 1 (required)**
- [ ] `platform-workshop.00_shared` schema exists
- [ ] `financial_docs_raw` volume contains PDFs for all 5 companies
- [ ] `financial_docs_sample` volume contains 2–3 sample PDFs
- [ ] `financial_docs_bronze`, `_silver`, `_gold`, `company_ai_investment_summary`, and `financial_docs_for_search` tables exist and have rows
- [ ] `financial_intelligence_pipeline` and `financial_intelligence_job` exist in LakeFlow Jobs
- [ ] `account users` have USE CATALOG, USE SCHEMA, SELECT, READ VOLUME
- [ ] `SETUP-Robust-AI-data-platforms-workshop` notebook is published to `/Workspace/Shared/`

**Session 2 (required only if running Session 2)**
- [ ] Vector Search endpoint `financial-docs-vs-endpoint` is online and in `ONLINE` state
- [ ] Vector Search index `platform-workshop.00_shared.financial_docs_for_search_index` is synced and `ready=True`
- [ ] Knowledge Assistant `financial-intelligence-assistant` is deployed and responding to test queries
- [ ] Graph Explorer app is deployed and accessible; 
  - `GRAPH_EXPLORER_URL` is updated in `04_hybrid_context_layers.ipynb`
  -    All users have CAN USE on the warehouse
  -    Table set to something that doesn't exist
  -    App will need to be given access to the users' edge tables as part of the workshop steps.